# 32장 · 검토 과정을 펼쳐 볼 수 있는 문의 챗봇 완성하기

회원은 지금 해야 할 일을 알고 싶고, 동료 담당자는 왜 그 답이 나왔는지 확인하고 싶습니다. 한 번의 입력에 대해 답변과 작업 기록을 함께 만들고, 화면이 다시 그려질 때는 이미 저장한 결과를 읽습니다. 짝이 보낸 새 문의의 실행을 함께 추적해 보세요.

## 회원 화면과 담당자 화면의 읽는 순서

회원은 “지금 안내 데스크로 가도 되는가?”를 가장 먼저 확인합니다. 담당자는 오류가 생겼을 때 어떤 규정을 읽고 무엇을 물었는지 거슬러 올라가야 합니다. 종이에 사용자 질문, 최종 답, 최초 초안, 평가 의견, 종료 사유를 적고 회원에게 늘 보여 줄 항목과 눌렀을 때 보여 줄 항목을 정하세요. 상한에서 끝난 경우만큼은 담당자 확인이 필요하다는 점이 답을 읽기 전에 보여야 합니다. 다음 화면의 배치를 그 판단과 대조해 보세요.

준비물은 실습 묶음의 `pre-mid/documents/sources/loan-rules.md`와 `pre-mid/documents/questions/loan-inquiries.md`입니다. 짝과 검토할 기록지는 `pre-mid/documents/workflow/workshop.md`에 있습니다.

## 서비스에 넣을 세 종류의 문의

신청 날짜가 빠진 문의에는 추가 질문 후 멈춥니다. 일반 문의는 규정에서 답변 초안을 작성하고, 안내문 검토 문의는 사용자가 제출한 원문을 바로 검토 대상으로 삼습니다. 두 경로 모두 평가와 수정 상한을 적용합니다. 이번 앱에서는 네 조항을 한 번에 읽을 수 있어 규정 전체를 넘깁니다. 자료가 여러 권이라면 어느 문단을 근거로 줄지 다시 결정해야 합니다.

## 실습 준비

8장에서 모델을 호출했던 Jupyter 커널을 선택하고, 이 장의 실습 묶음을 풀어 노트북과 pre-mid 폴더를 나란히 둡니다. 새로운 커널이라면 빈 셀에서 `%pip install -U openai`를 실행하고 수업용 API 키를 환경 변수로 설정한 뒤 시작하세요. 키는 노트북 셀이나 제출물에 적지 않습니다. 아래 함수에는 매번 지시와 이번에 읽을 자료를 전달합니다. 출력은 실행할 때마다 표현이 달라질 수 있습니다.

In [ ]:
import os
from getpass import getpass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API 키: ")


In [ ]:
from pathlib import Path
import os
from openai import OpenAI

# ZIP을 푼 폴더에서 노트북을 열면 이 경로에 네 가지 대여 조항이 있습니다.
rules = Path("pre-mid/documents/sources/loan-rules.md").read_text(encoding="utf-8")
model = os.environ.get("OPENAI_MODEL", "gpt-4.1-mini")
client = OpenAI(timeout=45, max_retries=1)  # 8장에서 준비한 환경 변수 OPENAI_API_KEY를 읽습니다.

def ask(instruction, material):
    response = client.responses.create(
        model=model, instructions=instruction, input=material, store=False
    )
    return response.output_text

In [ ]:
route_instruction = 'answer, review, clarify 중 작업 이름 하나만 쓰세요. 규정 질문은 answer, 사용자가 쓴 초안의 검토는 review, 신청 대행 요청에 기종 또는 정확한 날짜가 빠졌으면 clarify입니다. 규정 질문에는 신청 정보가 없어도 됩니다.'
judge_instruction = '규정·회원 문의·초안을 대조하세요. 회원이 물은 모든 행동에 필요한 조건이 있고\n규정과 일치하면 첫 줄에 통과, 아니면 수정이라고 쓰세요. 다음 줄에는\n원문 조건과 초안에서 빠지거나 잘못된 구절을 짝지어 한두 문장으로 설명하세요.\n문의에서 묻지 않은 주변 규정을 추가하라고 요구하지 마세요.'
revise_instruction = '회원에게 보낼 초안을 고치세요. 검토 의견을 규정과 대조하고, 맞는 조건은 유지하세요. 설명 없이 수정한 안내만 쓰세요.'

## 입력과 출력을 묶어서 기록하기

기록에는 단계 이름과 실제 모델에 준 본문, 돌아온 텍스트가 한 쌍으로 들어갑니다. 프로그램이 분류한 경로와 마지막 종료 사유도 남깁니다. 화면을 갱신해도 새 호출을 만들지 않으려면 이 기록을 어디서 만들고 어디서 읽을지 분리해야 합니다.

In [ ]:
def process(message):
    trace = []
    def run_step(name, instruction, material):
        # 모델을 호출한 한 건당 기록 하나가 생깁니다.
        reply = ask(instruction, material)
        trace.append({"단계": name, "지시": instruction, "입력": material, "출력": reply})
        return reply

    route = run_step("문의 분류", route_instruction, message).strip().lower()
    if route not in {"answer", "review", "clarify"}:
        raise ValueError("분류 결과 확인 필요: " + route)
    if route == "clarify":
        reply = run_step("추가 질문", "신청자가 주지 않은 정보만 물으세요. 예약을 완료하지 마세요.", rules + "\n" + message)
        return {"경로": route, "답변": reply, "종료": "정보 대기", "수정 횟수": 0, "기록": trace}

    if route == "review":
        # 원문의 잘못된 표현을 고치기 전에 그대로 남깁니다.
        draft = message
    else:
        notes = run_step("요구 추출", "회원 사실과 질문을 빠짐없이 메모하세요. 날짜와 수량을 보존하세요.", message)
        draft = run_step("초안 작성", "규정과 메모를 읽고 짧게 답하세요. 근거 조항을 적으세요.", rules + "\n" + notes)
    first_draft, edits = draft, 0
    while True:
        context = rules + "\n문의\n" + message + "\n검토 대상 초안\n" + draft
        raw = run_step("기준 대조", judge_instruction, context)
        verdict, _, reason = raw.strip().partition("\n")
        if verdict.strip() not in ("통과", "수정") or not reason.strip():
            raise ValueError("검토 형식 확인 필요: " + raw)
        if verdict.strip() == "통과":
            stop = "기준 충족"
            break
        if edits >= 1:
            stop = "수정 상한"
            break
        draft = run_step("초안 수정", revise_instruction, context + "\n검토 의견\n" + reason)
        edits += 1

    if route == "review" and edits == 0:
        # 초안이 그대로 통과해도 회원에게는 검토 결과임을 알립니다.
        draft = "검토 결과: " + reason
    return {"경로": route, "답변": draft, "최초 초안": first_draft,
            "종료": stop, "수정 횟수": edits, "기록": trace}

## 화면을 띄우기 전에 새 문의로 함수 시험하기

마감 뒤 바로 수령할 수 있는지 묻는 문장을 보내고 R1과 R2로 결과를 판정합니다. 모델이 사정을 고려해 예외를 만들어 주면 규정과 어긋나는 답입니다. 그때는 평가 의견과 실제 초안을 함께 읽어 어떤 단계가 예외를 받아들였는지 찾습니다.

In [ ]:
new_message = 'B형을 내일 쓰려고 하는데 아직 신청 전이에요. 지금은 오후 5시 반이에요. 내일 바로 안내 데스크로 가면 될까요?'
example = process(new_message)
print(example["경로"], example["종료"], example["수정 횟수"])
print(example["답변"])
for item in example["기록"]:
    print(item["단계"])

이 문의의 목표는 신규 예약 처리보다 바로 수령 가능한지 규정을 확인하는 것입니다. 질문의 표현 때문에 clarify가 선택되었으면 모델의 분류 지시를 다시 읽으세요. 마감 경과와 미승인을 근거로 수령할 수 없다고 답하는지 사람이 직접 확인합니다.

## 사용자 입력과 화면 다시 그리기 분리하기

Streamlit은 입력을 보내거나 위젯을 바꿀 때 스크립트를 위에서 다시 실행합니다. 이미 끝난 작업은 세션 상태에 두고 읽습니다. 실제 작업은 새 발화가 들어온 경우에만 실행합니다. 추가 정보를 되묻는 경로에서는 다음 발화를 원문의 보충 정보로 보냅니다. 다른 새 문의로 시작하려면 새 대화 버튼을 누르세요.

In [ ]:
app_screen = r'''
import streamlit as st

st.title("녹음기 대여 문의")
if "turns" not in st.session_state:
    st.session_state.turns = []
    st.session_state.pending = ""

if st.button("새 대화"):
    st.session_state.turns = []
    st.session_state.pending = ""

show_trace = st.checkbox("실행 기록 보기", value=True)
message = st.chat_input("회원 문의나 검토할 안내를 입력하세요")
if message:
    # 추가 질문에 대한 답이면 이전 문의까지 한 입력으로 복원합니다.
    pending = st.session_state.pending
    current = pending + "\n추가로 알려 준 정보\n" + message if pending else message
    result = process(current)
    st.session_state.turns.append((message, result))
    st.session_state.pending = current if result["경로"] == "clarify" else ""

for text, result in st.session_state.turns:
    st.chat_message("user").write(text)
    with st.chat_message("assistant"):
        if result["종료"] == "수정 상한":
            st.warning("담당자 검토가 필요한 초안입니다.")
        st.write(result["답변"])
        if show_trace:
            with st.expander("초안과 검토 과정"):
                st.write(result["경로"], result["종료"], "수정", result["수정 횟수"], "회")
                if "최초 초안" in result:
                    st.write(result["최초 초안"])
                for item in result["기록"]:
                    st.write(item["단계"])
                    st.json(item)
'''

상태에 쌓인 것은 보기용 문자열만이 아닙니다. 같은 문의에 어떤 입력과 출력이 있었는지도 남습니다. 체크박스를 켰다 끈 뒤에도 호출 기록의 행 수, 종료 사유, 답의 표현이 변하지 않아야 합니다. 최초 초안에는 정정되기 전 문장이 남아 있어야 합니다.

## 실제 채팅 화면에서 실행하기

노트북 이름을 chapter32.ipynb로 유지하고 저장하세요. 아래 셀이 현재 파일에서 수업 중 편집한 함수 셀만 꺼내 화면과 함께 workflow_app.py로 저장합니다. 터미널을 같은 폴더에서 열어 `python -m pip install streamlit openai`로 준비하고 `python -m streamlit run workflow_app.py`를 실행하세요. 터미널에도 8장에서 정한 OPENAI_API_KEY가 설정돼 있어야 합니다. 앱을 끝낼 때는 터미널에서 Ctrl+C를 누릅니다.

In [ ]:
import json

notebook = json.loads(Path("chapter32.ipynb").read_text(encoding="utf-8"))
sections = ["".join(cell["source"]) for cell in notebook["cells"]
            if "workflow-source" in cell["metadata"].get("tags", [])]
Path("workflow_app.py").write_text("\n\n".join(sections + [app_screen]), encoding="utf-8")
print("workflow_app.py를 저장했습니다.")

## 실습 · 짝의 문의 한 건을 끝까지 인계하기

개인·동료 실습 120분. 한 사람은 신청 규정에 익숙하지 않은 회원, 한 사람은 담당자를 맡습니다. 회원은 Q3을 자기 말로 보내고 날짜와 기종을 추가합니다. 이어 새 대화에서 Q2 또는 Q5의 날짜·시간을 바꿔 묻습니다. 담당자는 화면만 보고 선택 경로, 최초 초안, 고친 조건과 종료 사유를 짚습니다. 체크박스 전후의 기록 건수를 적고 새 대화 후 빈 화면도 확인합니다. 모델이 선택한 경로를 그대로 정답으로 기록하지 말고 본인이 예상한 이유를 한 줄 붙입니다.

In [ ]:
message = "내일이 대여일인데 신청은 오늘 오후 5시까지인가요?"
# 이 질문은 날짜를 묻는 것이 아니라 신청 마감 규정에 대한 질문입니다.

<details class="notebook-answer"><summary>모범 풀이와 대조 기준</summary>
<pre><code>message = &quot;내일이 대여일인데 신청은 오늘 오후 5시까지인가요?&quot;
result = process(message)
print(result[&quot;경로&quot;], result[&quot;종료&quot;], result[&quot;답변&quot;])
print(len(result[&quot;기록&quot;]))</code></pre>
<p>예시 기록. “이번 달에 하나 빌려 주세요”에는 빠진 기종과 정확한 날짜를 물어 정보 대기에서 끝납니다. “내일 쓰려는데 오늘 5시 반입니다”에는 R1 마감과 R2 승인 조건을 확인합니다. 화면의 체크박스를 바꿔도 이전 문의의 모델 호출 기록은 늘지 않습니다. 검토가 상한에서 끝나면 답 위에 담당자 검토 표시가 있어야 합니다.</p>
</details>

## 다음 작업에 가져갈 기록

동료가 실제로 보낸 한 문장과 예상 경로·실행 경로, 다르면 처음 어긋난 단계를 적어 두세요. 33장에서는 Python 함수 안의 상태와 다음 단계 선택을 LangGraph의 상태·노드·간선으로 옮깁니다.

[Streamlit 세션 상태](https://docs.streamlit.io/develop/api-reference/caching-and-state/st.session_state) · [OpenAI Responses API](https://platform.openai.com/docs/api-reference/responses/create)